# TelcoTR - Müşteri Kaybı (Churn) Tahmin Analizi

**Amaç:** Müşteri geçmiş verisinden yola çıkarak, bir müşterinin ayrılıp ayrılmayacağını (`Churn`) önceden tahmin eden bir model kurmak; modelin ne kadar güvenilir olduğunu dürüstçe ölçmek ve sonuçları iş diliyle raporlamak.

**Akış:** Keşif (EDA) → Veri kalitesi kontrolü ve temizlik → Train/Test ayrımı → Ön işleme (leakage'sız) → İki model eğitimi → Metrik seçimi ve karşılaştırma → Hata analizi → Sonuç.


## 1. Kütüphaneler ve veri yükleme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay, precision_recall_curve
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv("data/telco.csv")

In [ ]:
print("Boyut:", df.shape)

In [ ]:
df.head()

## 2. İlk keşif - veri ne durumda?

Modele girmeden önce her columnun **tipini**, **eksik/bozuk değerlerini** ve **tekilliğini** kontrol ediyorum.


In [ ]:
df.info()

In [ ]:
# customerID feature olarak kullanılmayacak olsa da gerçekten unique mi diye kontrol ediyorum.

print("Satır sayısı :", len(df))
print("Benzersiz ID :", df["customerID"].nunique())
print("Tam kopya satır :", df.duplicated().sum())

In [ ]:
# isna() ile eksik değer var mı diye kontrol ediyorum.

df.isna().sum()[df.isna().sum() > 0]

In [ ]:
# isna() kontrolü temiz çıksa da TotalCharges kolonu sayısal olması gerekirken object çıktığı için derinlemesine kontrol yapıyorum.
# TotalCharges'ı sayıya çevirmeye çalışalım ve bozuk olanları yakalayayım.

total_charges_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
invalid_mask = total_charges_numeric.isna()
print("Total Charges'ta sayıya çevrilemeyen bozuk satır sayısı :", invalid_mask.sum())
df.loc[invalid_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

**Bulgu:** 11 satırda `TotalCharges` değeri `' '` olarak doldurulmuş ve tamamı *`tenure=0`*. Yani bu müşteriler henüz bir tam ay doldurmamış. Sonuç olarak bunun "kayıp veri" değil, "henüz oluşmamış veri" olduğunu değerlendiriyorum. Mantıksal olarak `tenure=0` olduğundan `TotalCharges` değerlerini `0` ile dolduruyorum.


In [ ]:
# Categorical columnsda beklenmedik değer / yazım tutarsızlığı var mı?

categorical_cols = df.select_dtypes(include="object").columns.drop(["customerID", "TotalCharges"])

for c in categorical_cols:
    print(f"{c:20} -> {sorted(df[c].unique())}")

Kategorik kolonlar veri sözlüğüyle birebir uyumlu; büyük/küçük harf ya da boşluk kaynaklı bir bozukluk yok.

## 3. Hedef değişken: `Churn`

README'de zaten belirtilmiş: hedef dengeli değil. Ne kadar dengesiz olduğunu görmek istiyorum -
çünkü bu, ileride hangi metriği kullanacağımı doğrudan belirleyecek.


In [ ]:
churn_distribution = df["Churn"].value_counts()
churn_ratio = df["Churn"].value_counts(normalize=True)
print(churn_distribution)
print((churn_ratio * 100).round(2).astype(str) + " %")

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
churn_distribution.plot(kind="bar", color=["#4C72B0", "#C44E52"], ax=ax)
ax.set_title("Churn Dağılımı")
ax.set_xlabel("")
ax.set_ylabel("Müşteri Sayısı")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Bulgu:** Müşterilerin yaklaşık **%73,5'i** kalıyor (No), **%26,5'i** ayrılıyor (Yes). Bu dengesiz bir sınıflandırma olduğundan accuracy tek başına yanıltıcı bir metrik olacaktır. Bu yüzden recall, precision ve ROC-AUC gibi metriklere daha çok güveneceğim.


## 4. Sayısal değişkenlerin churn'e göre dağılımı

Modele geçmeden önce, sezgisel olarak hangi değişkenlerin churn ile ilişkili görünebileceğine bakmak
faydalı - hem veriyi anlamak hem de sonuçta modelin bulduklarıyla kıyaslamak için.

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col in zip(axes, numeric_cols):
    for churn_value, color in [("No", "#4C72B0"), ("Yes", "#C44E52")]:
        subset = df.loc[df["Churn"] == churn_value, col]
        ax.hist(subset, bins=30, alpha=0.6, label=churn_value, color=color, density=True)
    ax.set_title(col)
    ax.legend(title="Churn")
plt.tight_layout()
plt.show()

**Bulgu:** Ayrılan müşteriler (Yes) belirgin biçimde düşük tenure tarafında yoğunlaşıyor. Yani şirkette az süredir olan müşteriler daha çok ayrılıyor. Ve yüksek MonthlyCharges tarafına doğru bir kayma var. Bu ikisi muhtemelen modelde güçlü sinyaller olacak.

## 5. Temizlik - kararları uygulama

Yukarıdaki keşfe dayanarak:
1. `TotalCharges`'ı sayısala çevirip 11 bozuk satırı `0` ile dolduruyorum (gerekçe: yukarıda açıklandı).
2. `customerID`'yi düşürüyorum — modele hiçbir bilgi katmıyor, yalnızca benzersiz bir etiket.
3. Hedefi (`Churn`) `0/1`'e çeviriyorum.


In [ ]:
clean_df = df.copy()
clean_df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
clean_df = clean_df.drop(columns=["customerID"])

X = clean_df.drop(columns=["Churn"])
y = (clean_df["Churn"] == "Yes").astype(int)

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical_cols = [c for c in X.columns if c not in numeric_cols]
print("Numeric columns: ", numeric_cols)
print("Categorical columns: ", categorical_cols)

## 6. Train / Test ayrımı — leakage'ı en baştan engelleme

**Neden önce split, sonra her şey?** Ölçekleme (`StandardScaler`) veya kodlama (`OneHotEncoder`) gibi
"veriden istatistik öğrenen" adımları tüm veri üzerinde yaparsak, test setinin bilgisi dolaylı olarak
eğitime sızar (data leakage) - model gerçekte göremeyeceği bir bilgiden faydalanmış olur ve sonuçlar
gerçek dünyada tekrar etmeyecek kadar iyimser çıkar.

**Neden `stratify=y`?** Hedef dengesiz (%26,5 churn). Rastgele bir split, şans eseri train ve test'te
farklı churn oranları üretebilir. `stratify` bu oranı her iki sette de aynı tutar, kıyaslamayı adil kılar.

Ön işleme adımlarını (`ColumnTransformer`) bir `Pipeline` içine koyuyorum; böylece `fit` yalnızca
`X_train` üzerinde, `transform` ise hem train hem test'e otomatik ve doğru sırada uygulanıyor.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train :", X_train.shape, '| Test:', X_test.shape)
print(f"Train churn oranı: {y_train.mean():.4f} | Test churn oranı: {y_test.mean():.4f}")

In [ ]:
# Ön işleme: sayısalları ölçekle, kategorikleri one-hot encode et.
# drop="first": dummy variable trap'i önlemek için her kategorik kolonun bir seviyesini referans olarak bırak.

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_cols)
])

## 7. İki model, adil karşılaştırma

**Neden bu iki model?**
- **Lojistik Regresyon:** basit, hızlı, katsayıları doğrudan yorumlanabilir ("hangi faktör churn'ü
  ne yönde etkiliyor" sorusuna doğrudan cevap verir). İş tarafına açıklama yaparken güçlü bir avantaj.
- **Random Forest:** doğrusal olmayan ilişkileri ve değişkenler arası etkileşimleri yakalayabilir;
  genelde biraz daha yüksek performans verir ama yorumlanabilirliği katsayılar kadar doğrudan değildir
  (yine de `feature_importances_` ile hangi değişkenlerin öne çıktığını görebilirim).

**Adil karşılaştırma için:** İkisini de aynı `X_train`/`X_test` split'i, aynı ön işleme pipeline'ı ve
aynı metrik setiyle değerlendiriyorum. `class_weight='balanced'` her iki modelde de kullanıldı -
azınlık sınıfına (churn=Yes) modelin daha fazla önem vermesini sağlıyor; aksi halde model "herkese
No de, zaten %73 doğru olursun" tembelliğine kayabilir.

Önce train seti üzerinde **5-fold stratified cross-validation** ile modellerin ne kadar istikrarlı
olduğuna bakıyorum (tek bir split'in şans eseri iyi/kötü çıkmasını azaltmak için), sonra hiç
görmedikleri test setinde final değerlendirmeyi yapıyorum.


In [ ]:
models = {
    "Lojistik Regresyon": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=400, max_depth=8, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model)
    ])
    
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision", "recall", "precision"]
    )
    cv_results[name] = scores
    
    print(f"== {name} - 5-fold CV (train seti üzerinde) ==")
    
    for metric in ["test_roc_auc", "test_average_precision", "test_recall", "test_precision"]:
        print(f"{metric:24}: {scores[metric].mean():.4f} (+/- {scores[metric].std():.4f})")
    print()

In [ ]:
# Final modelleri tüm train setiyle eğitip test setinde değerlendiriyorum

fitted_pipelines = {}
test_probabilities = {}
test_predictions = {}

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    test_probabilities[name] = pipe.predict_proba(X_test)[:, 1]
    test_predictions[name] = pipe.predict(X_test)
    
    print(f"== {name} - Test seti sonuçları ==")
    print("ROC-AUC :", round(roc_auc_score(y_test, test_probabilities[name]), 4))
    print("PR-AUC :", round(average_precision_score(y_test, test_probabilities[name]), 4))
    print(classification_report(y_test, test_predictions[name], digits=3, target_names=["Kalır (No)", "Ayrılır (Yes)"]))
    print()

## 8. Neden accuracy değil? - Doğru metrik tartışması

Önce yanıltıcı örneği somut sayıyla gösterelim: "herkese kalır de" diyen saf (naif) bir model bile
sırf hedef dengesiz olduğu için yüksek accuracy alır.

In [ ]:
naive_accuracy = (y_test == 0).mean()

print(f"""
    Naif 'herkes kalır' tahmininin accuracy'si: {naive_accuracy:.4f} .
    Bu model hiçbir öngörü gücüne sahip değil. Sadece çoğunluk sınıfını tekrarlıyor.
    Yukarıdaki gerçek modellerin accuracy'si de bu naif değere yakın bir bantta olabilir.
    Ama churn sınıfı için precision/recall onları netçe ayırıyor.
    """)

**Neden accuracy yanıltıcı?** %73,5 "kalır" olan bir veri setinde, hiçbir şey öğrenmeyen bir model bile
yalnızca çoğunluk sınıfını tahmin ederek yaklaşık bu oranda "doğru" görünür. Yani accuracy tek başına
modelin gerçekten öngörü gücü olup olmadığını ayırt etmiyor.

**Bu iş problemi için hangi metrikler daha anlamlı?**
- **Recall (churn=Yes için):** Gerçekten ayrılacak müşterilerin kaçını yakalıyoruz? Bunu kaçırmak
  (false negative) = elimden kayan bir müşteri, hiçbir müdahale şansı olmadan gidiyor. İş açısından
  genelde en pahalı hata türü budur.
- **Precision (churn=Yes için):** "Ayrılacak" dediklerimizin kaçı gerçekten ayrılıyor? Düşükse
  pazarlama, ayrılmayacak müşterilere boşuna elde tutma teklifi/indirim harcıyor (false positive maliyeti
  — genelde false negative'den daha ucuz ama sıfır değil).
- **ROC-AUC:** Eşikten bağımsız olarak modelin pozitif/negatif sınıfı genel olarak ne kadar iyi ayırt
  ettiğini ölçer. Dengesiz veride bile makul bir genel performans göstergesidir.
- **PR-AUC (average precision):** Azınlık sınıfına (churn) odaklandığı için dengesiz veri setlerinde
  ROC-AUC'dan bazen daha bilgilendiricidir; ikisini birlikte raporluyorum.

Bu proje için özellikle **recall**'a öncelik veriyorum çünkü kaçırılan bir churn'ün iş maliyeti,
yanlış alarmın maliyetinden yüksek - ama precision'ı da tamamen göz ardı etmiyorum (aşırı düşük
precision, pazarlama bütçesini israf eder).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for name in models:
    RocCurveDisplay.from_predictions(y_test, test_probabilities[name], name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, test_probabilities[name], name=name, ax=axes[1])

axes[0].set_title("ROC Eğrisi")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.5)
axes[1].set_title("Precision-Recall Eğrisi")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", alpha=0.5, label="Naif taban çizgi")
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Model karşılaştırma tablosu ve final model seçimi

In [ ]:
summary_rows = []

for name in models:
    summary_rows.append({
        "Model": name,
        "ROC-AUC": round(roc_auc_score(y_test, test_probabilities[name]), 4),
        "PR-AUC": round(average_precision_score(y_test, test_probabilities[name]), 4),
        "Recall (churn)": round(
            classification_report(y_test, test_predictions[name], output_dict=True)["1"]["recall"], 4
        ),
        "Precision (churn)": round(
            classification_report(y_test, test_predictions[name], output_dict=True)["1"]["precision"], 4
        ),
        "Accuracy": round(
            classification_report(y_test, test_predictions[name], output_dict=True)["accuracy"], 4
        )
    })
    
summary_df = pd.DataFrame(summary_rows).set_index("Model")
summary_df

**Karar:** İki model de ~0,84 ROC-AUC ile hedefin (0,80) üzerinde ve birbirine oldukça yakın.
Recall değerleri **aynı** çıkıyor (iki model de churn edenlerin aynı oranını yakalıyor), ama
**Random Forest daha az false positive üretiyor** — yani aynı recall'u daha az "boşa alarm" ile
sağlıyor, bu da precision'ı ve genel accuracy'yi yukarı çekiyor. Bu yüzden **final model olarak
Random Forest'ı** seçiyorum; Lojistik Regresyon'u ise katsayılarının kolay yorumlanabilir olması
sebebiyle "neden" sorusuna cevap vermek için yanında tutuyorum.


## 10. Hangi değişkenler churn'ü sürüklüyor?

In [ ]:
# Lojistik Regresyon coefficient'ları -- işaret ve büyüklük doğrudan yorumlanabilir

logreg_pipe = fitted_pipelines["Lojistik Regresyon"]

feature_names = logreg_pipe.named_steps["prep"].get_feature_names_out()
coefficients = logreg_pipe.named_steps["model"].coef_[0]

coefficient_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
}).sort_values("coefficient")

print("--- Churn olasılığını EN ÇOK AZALTAN 6 feature ---")
print(coefficient_df.head(6).to_string(index=False))
print()

print("--- Churn olasılığını EN ÇOK ARTIRAN 6 feature ---")
print(coefficient_df.tail(6).to_string(index=False))

In [ ]:
# Random Forest feature importance -- büyüklük sıralaması (işaret bilgisi yok ama etkileşimleri de yakalar)

rf_pipe = fitted_pipelines["Random Forest"]

rf_feature_names = rf_pipe.named_steps["prep"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(7, 5))

importance_df.head(10).set_index("feature")["importance"].sort_values().plot(
    kind="barh",
    ax=ax,
    color="#4C72B0"
)

ax.set_title("Random Forest - en önemli 10 feature")

plt.tight_layout()
plt.show()

**Yorum:** İki model de tutarlı bir hikâye anlatıyor:
- **Sözleşme türü** en güçlü koruyucu faktör - `Two year` ve `One year` sözleşmesi olanlar çok daha az ayrılıyor.
  Mantıklı: uzun sözleşme zaten müşteriyi bağlıyor.
- **`tenure`** (ne kadar süredir müşteri) churn'ü azaltıyor - yeni müşteriler daha kırılgan.
- **Fiber optic internet** churn'ü artıran en güçlü tekil sinyallerden biri - muhtemelen fiyat/hizmet
  memnuniyetsizliğine işaret ediyor, bu iş tarafı için araştırılmaya değer bir bulgu.
- **`Electronic check`** ile ödeme yapanlar diğer ödeme yöntemlerine göre daha çok ayrılıyor -
  bu ödeme yönteminin otomatik olmaması (bankadan/kartan otomatik değil) ile ilişkili olabilir.

## 11. Hata analizi - model nerede yanılıyor, işe maliyeti ne?

En kritik hata türü **false negative**: gerçekte ayrılacak bir müşteriyi model "kalır" diye
işaretliyor. Bu, o müşteriye hiçbir elde tutma müdahalesi (indirim, arama, özel teklif) gitmeyeceği
anlamına geliyor - yani **doğrudan kayıp bir müşteri**. Final modelim olan Random Forest üzerinden
bu grubu inceliyorum.


In [ ]:
error_df = X_test.copy()
error_df["actual"] = y_test.values
error_df["prediction"] = test_predictions["Random Forest"]
error_df["probability"] = test_probabilities["Random Forest"]

cm = confusion_matrix(y_test, test_predictions["Random Forest"])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["Kalır", "Ayrılır"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Random Forest - Confusion Matrix (test seti)")
plt.tight_layout()
plt.show()

false_negative = error_df[(error_df["actual"] == 1) & (error_df["prediction"] == 0)]
print(f"Kaçırılan (false negative) müşteri sayısı: {len(false_negative)} / test setindeki {y_test.sum()} gerçek churn")
print(f"Yani gerçek churn'lerin {100 * len(false_negative) / y_test.sum():.1f}%'i kaçırılıyor.")

In [ ]:
print("Kaçırılan (FN) grupta Contract dağılımı:")
print((false_negative["Contract"].value_counts(normalize=True) * 100).round(1))
print()
print("Test setindeki TÜM gerçek churn müşterilerinde Contract dağılımı:")
print((error_df[error_df["actual"] == 1]["Contract"].value_counts(normalize=True) * 100).round(1))

**Bulgu - önemli bir kör nokta:** Test setindeki gerçek churn'lerin **%88'i** `Month-to-month`
sözleşmeli, ama kaçırdığım (FN) grupta bu oran **%49'a düşüyor** - yani model, **`One year` ve
`Two year` sözleşmeli ama yine de ayrılan** müşterileri orantısız biçimde kaçırıyor. Bunun mantığı var:
model "uzun sözleşme = düşük risk" örüntüsünü genel olarak doğru öğrenmiş, ama bu örüntüye **uymayan**
istisnai müşterileri (uzun sözleşmeli olup yine de ayrılanları) tam olarak yakalayamıyor. İş tarafına
vereceğim somut mesaj: *"Uzun sözleşmeli müşteriler genelde güvenli ama modelin gözden kaçırma riski
en yüksek olduğu grup da tam olarak burası - bu segmentte ekstra bir uyarı sinyaline ihtiyaç var."*


In [ ]:
print("Kaçırılan (FN) grupta model olasılık tahmini (0.5 eşiğine ne kadar yakınlar):")
print(false_negative["probability"].describe())

**Bulgu:** Kaçırılan müşterilerin olasılık tahminleri **ortalama ~0,31**, çoğu 0,5 eşiğinin
hemen altında (75. yüzdelik dilim ~0,43'e kadar çıkıyor). Yani bunların önemli bir kısmı "az farkla
kaçan", eşiğe yakın vakalar. Bu, **eşiği düşürerek** (örn. 0,5 yerine 0,35) daha fazla gerçek churn'ü
yakalayabileceğimizi gösteriyor - tabii bunun bedeli daha fazla false positive (precision düşüşü)
olacak. Bu, iş tarafının karar vermesi gereken bir **maliyet dengesi**: bir churn'ü kaçırmanın maliyeti
(kaybedilen müşterinin yaşam boyu değeri) ile boşa giden bir elde tutma teklifinin maliyeti
karşılaştırılarak optimal eşik seçilebilir.

In [ ]:
# Eşiği değiştirmenin precision / recall üzerindeki etkisini görselleştir

precision, recalls, thresholds = precision_recall_curve(y_test, test_probabilities["Random Forest"])

fig, ax =  plt.subplots(figsize=(7, 4.5))

ax.plot(thresholds, precision[:-1], label="Precision", color="#4C72B0")
ax.plot(thresholds, recalls[:-1], label="Recall", color="#C44E52")
ax.axvline(0.5, linestyle="--", color="gray", alpha=0.6, label="Varsayılan eşik (0.5)")
ax.set_xlabel("Karar eşiği")
ax.set_ylabel("Değer")
ax.set_title("Eşik değiştikçe Precision / Recall değişimi (Random Forest)")
ax.legend()
plt.tight_layout()
plt.show()

## 12. Özet ve sonuç

- **Veri kalitesi:** `TotalCharges` columnunda 11 satırlık, açıklanabilir bir bozukluk (`tenure=0` yeni
  müşteriler) bulundu ve mantıklı bir varsayımla (`0` ile doldurma) düzeltildi. `customerID` modelden
  çıkarıldı. Başka ciddi bir veri kalitesi sorunu tespit etmedim.
- **Hedef dengesizliği:** %73,5 kalır / %26,5 ayrılır - bu yüzden accuracy yerine recall/precision/
  ROC-AUC/PR-AUC odaklı değerlendirme yaptım.
- **Modelleme:** Lojistik Regresyon ve Random Forest, aynı train/test split ve aynı (leakage'sız)
  ön işleme pipeline'ı ile adil şekilde karşılaştırıldı. Her ikisi de ~0,84 ROC-AUC ile hedefin
  (0,80) üzerinde. Random Forest, aynı recall'u daha az false positive ile sağladığı için final
  model olarak seçildi.
- **En güçlü sinyaller:** Sözleşme türü, tenure, aylık ücret, fiber optic internet ve elektronik
  çek ile ödeme.
- **En kritik zafiyet:** Model, uzun sözleşmeli ama yine de ayrılan "beklenmedik" churn'leri
  orantısız biçimde kaçırıyor; bu segment için ek bir izleme/uyarı mekanizması öneriyorum.

İş diliyle yazdığım tam öneriler için bkz. **`RAPOR.md`**. Yeniden üretilebilir, tek komutla
çalışan final akış için bkz. **`train.py`**.
